In [17]:
import re
import os
import numpy as np
from astropy.io import ascii
from astropy.io import fits
from astropy.table import Table
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO
from matplotlib.ticker import AutoMinorLocator
from astropy.constants import L_sun
from matplotlib import gridspec
import astropy.units as u
from math import pi
import gc
from matplotlib.ticker import ScalarFormatter
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from astropy.table import Column

#### Creating File (once)

In [273]:
fits_name = 'modulus_tracker_cepheids.fits'
fits_name = 'modulus_tracker_trgb.fits'
fits_name = 'modulus_tracker.fits'

#Making primary

rands_x1hdu = np.random.random((1,1))
primary_hdu = fits.PrimaryHDU(data=rands_x1hdu)

HDU = fits.HDUList([primary_hdu])



hosts = ['LMC','IC0010','MESSIER033','MESSIER081','MESSIER101','MRK0116','NGC0925','NGC2366',
         'NGC2403','NGC2541','NGC3198','NGC3319','MESSIER106','NGC4395','NGC6822','NGC1073',
         'NGC2500','NGC3184','MESSIER096','NGC3370','MESSIER066','NGC4214','NGC4414','NGC4496',
         'NGC4535','NGC4536','NGC4725','UGC08091','NGC5204','UGC09128','NGC5584']
len(hosts)


def binTable_creator(host,mode='C'):
    values = []
    HDR = fits.Header()
    if mode=='C':
        refcode = Column(values, name='RefCode', description="Article's RefCode", dtype='S40')
        citation = Column(values, name='Citation', description="Article's Citation", dtype='S40')
        band = Column(values, name='Band', description='Band', dtype='S40')
        modulus = Column(values, name='mu_0', description="Distance modulus", dtype='float64')
        err_rand = Column(values, name='e_R', description='Random error', dtype='float64')
        err_sys = Column(values, name='e_S', description='Sytematic error', dtype='float64')
        err_tot = Column(values, name='e_T', description='Total error', dtype='float64')
        category = Column(values, name='Category', description='Quality of observation given error', dtype='int64')
        Ncef = Column(values, name='Ncef', description='Number of cepheids reported', dtype='int64')
        Zcorr = Column(values, name='Zcorr', description='Metallicity correction', dtype='bool')
        zeroP = Column(values, name='zeroP', description='Zero-Point calibration', dtype='S20')
        rank = Column(values, name='Rank', description='Observation type (rank)', dtype='int64')
        year = Column(values, name='Year', description='Year of the publication', dtype='int64')
        comments = Column(values, name='Comments', description='Commentary', dtype='S800')

        T = Table([refcode, citation, band, modulus, err_rand, err_sys,err_tot,category,Ncef,Zcorr,zeroP,rank,year,comments])
        HDR.insert(0,('METHOD', 'CEPHEIDS', 'Cepheids PLR method'))

        hdu = fits.BinTableHDU(data = T,
                header=HDR,
                name = host)
        
        hdu_head = hdu.header
        hdu_head.set('EXTNAME', host, "Name of Host Galaxy")
        hdu_head.set('TTYPE1', 'RefCode', T['RefCode'].description)
        hdu_head.set('TTYPE2', 'Citation', T['Citation'].description)
        hdu_head.set('TTYPE3', 'Band', T['Band'].description)
        hdu_head.set('TTYPE4', 'mu_0', T['mu_0'].description)
        hdu_head.set('TTYPE5', 'e_R', T['e_R'].description)
        hdu_head.set('TTYPE6', 'e_S', T['e_S'].description)
        hdu_head.set('TTYPE7', 'e_T', T['e_T'].description)
        hdu_head.set('TTYPE8', 'Category', T['Category'].description)
        hdu_head.set('TTYPE9', 'Ncef', T['Ncef'].description)
        hdu_head.set('TTYPE10', 'Zcorr', T['Zcorr'].description)
        hdu_head.set('TTYPE11', 'zeroP', T['zeroP'].description)
        hdu_head.set('TTYPE12', 'Rank', T['Rank'].description)
        hdu_head.set('TTYPE13', 'Year', T['Year'].description)
        hdu_head.set('TTYPE14', 'Comments', T['Comments'].description)

        hdu_head.insert(39, ('COMMENT',''))
        hdu_head.insert(40, ('COMMENT',
                             'Category: (1) Explicitly random error considering purely photometric errors of all Cepheids; (2) Explicitly random and systematic error separately, the total error is calculated from equation X (mentioned in the manuscript); (3) Random error but all or most systematic errors are mentioned in the publication; (4) Total error only; (5) Random error but the distance modulus is expressed without adding the zero point. The distance modulus presented in the publication is added manually.'))
        
        hdu_head.insert(41, ('COMMENT',''))
        hdu_head.insert(42, ('COMMENT',
                             'zeroP: LMC corresponds to the Large Magellanic Cloud; GC corresponds to Galactic Cepheids; N4258 corresponds to NGC4258; TM corresponds to Theoretical Models.'))
        
        hdu_head.insert(43, ('COMMENT',''))
        hdu_head.insert(44, ('COMMENT',
                             'Rank: (1) They are space telescopes like HST, Spitzer or JWST, (2) Telescopes located at Hawai, Chile or Canary Islands (3) Any other terrestrial telescope'))
        


        return hdu
        
    if mode=='T':
        refcode = Column(values, name='RefCode', description="Article's RefCode", dtype='S40')
        citation = Column(values, name='Citation', description="Article's Citation", dtype='S40')
        band = Column(values, name='Band', description='Band', dtype='S40')
        modulus = Column(values, name='mu_0', description="Distance modulus", dtype='float64')
        err_rand = Column(values, name='e_R', description='Random error', dtype='float64')
        rank = Column(values, name='Rank', description='Observation type (rank)', dtype='int64')
        year = Column(values, name='Year', description='Year of the publication', dtype='int64')
        comments = Column(values, name='Comments', description='Commentary', dtype='S800')

        T = Table([refcode, citation, band, modulus, err_rand,rank,year,comments])

        HDR.insert(0,('METHOD', 'TRGB', 'Tip of the Red Giant Branch method'))

        hdu = fits.BinTableHDU(data = T,
                header=HDR,
                name = host)
        
        hdu_head = hdu.header
        hdu_head.set('EXTNAME', host, "Name of Host Galaxy")
        hdu_head.set('TTYPE1', 'RefCode', T['RefCode'].description)
        hdu_head.set('TTYPE2', 'Citation', T['Citation'].description)
        hdu_head.set('TTYPE3', 'Band', T['Band'].description)
        hdu_head.set('TTYPE4', 'mu_0', T['mu_0'].description)
        hdu_head.set('TTYPE5', 'e_R', T['e_R'].description)
        hdu_head.set('TTYPE6', 'Rank', T['Rank'].description)
        hdu_head.set('TTYPE7', 'Year', T['Year'].description)
        hdu_head.set('TTYPE8', 'Comments', T['Comments'].description)
        hdu_head.insert(29, ('COMMENT',''))
        hdu_head.insert(30, ('COMMENT',
                             'Rank: (1) They are space telescopes like HST, Spitzer or JWST, (2) Telescopes located at Hawai, Chile or Canary Islands (3) Any other terrestrial telescope'))
        

        return hdu
    
lista_HDUs = [primary_hdu]

for i in range(len(hosts)):

    lista_HDUs.append(binTable_creator(hosts[i],mode='C'))

    lista_HDUs.append(binTable_creator(hosts[i],mode='T'))

len(lista_HDUs)

HDU_final = fits.HDUList(lista_HDUs)


HDU_final.writeto(fits_name, overwrite=True)

In [275]:

Galaxy = 'MRK0116'
Method = 'CEPHEIDS'

hdu_encontrado = None

with fits.open('modulus_tracker.fits') as hdulist:
    
    # Iterar sobre cada HDU en la lista
    for hdu in hdulist:
        header = hdu.header
        
        # Aplicar los criterios de búsqueda
        # Verificar si 'EXTNAME' existe Y si 'TELESCOP' existe y coincide
        if ('EXTNAME' in header and header['EXTNAME'] == Galaxy) and \
           ('METHOD' in header and header['METHOD'] == Method):
            
            # Si ambos criterios coinciden, hemos encontrado nuestro HDU
            hdu_encontrado = hdu
            print(f"\nHDU número {hdulist.index(hdu)}.\n")
            Tabla = Table.read(hdu)
            #print(Tabla)
            # Puedes salir del bucle si solo esperas uno
            break 
    
    # 3. Usar el HDU encontrado
    if hdu_encontrado is not None:
        print("\nCabecera del HDU:\n")
        print(repr(hdu_encontrado.header)) # Imprime una representación completa de la cabecera
        # Aquí puedes acceder a los datos: hdu_encontrado.data
    else:
        print("\nNo se encontró ningún HDU que coincida con ambos criterios.")

Tabla


HDU número 11.


Cabecera del HDU:

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 1005 / length of dimension 1                          
NAXIS2  =                    0 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                   14 / number of table fields                         
METHOD  = 'CEPHEIDS'           / Cepheids PLR method                            
EXTNAME = 'MRK0116 '           / Name of Host Galaxy                            
TTYPE1  = 'RefCode '           / Article's RefCode                              
TFORM1  = '40A     '                                                    

RefCode,Citation,Band,mu_0,e_R,e_S,e_T,Category,Ncef,Zcorr,zeroP,Rank,Year,Comments
str40,str40,str40,float64,float64,float64,float64,int64,int64,bool,str20,int64,int64,str800


In [267]:
Galaxy = 'MRK0116'
Method = 'CEPHEIDS'

hdu_index = -1
tabla_modificada = None

# Abrimos el archivo en modo lectura para buscar y cargar datos
with fits.open('modulus_tracker.fits') as hdulist:
    print(f"Buscando HDU con EXTNAME={Galaxy} Y METHOD={Method}...")

    for i, hdu in enumerate(hdulist):
        header = hdu.header

        if ('EXTNAME' in header and header['EXTNAME'] == Galaxy) and \
           ('METHOD' in header and header['METHOD'] == Method):
            
            print(f"HDU encontrado en el índice {i}. Cargando datos para modificar.")
            hdu_index = i
            
            # Cargar los datos FITS en un objeto Table de Astropy para manipulación
            tabla_existente = Table(hdu.data)
            
            # Modificar la tabla: añadir una nueva fila
            nueva_fila = ['2007ApJ...667L.151A',
                          'Aloisi et al.(2007)',
                          'W(VI)',
                          31.38,
                          0.17,
                          np.nan,
                          np.nan,
                          1,
                          3,
                          False,
                          'LMC',
                          1,
                          2007,
                          'Una Cefeida usual y 2 con extrapolacion por periodos largos']
            tabla_existente.add_row(nueva_fila)
            tabla_modificada = tabla_existente
            
            break # Salir del bucle una vez encontrado

# --- 3. Sobrescribir el archivo con la tabla modificada ---

if hdu_index != -1 and tabla_modificada is not None:
    print("\nGuardando cambios...")

    # Abrimos el archivo de nuevo, pero esta vez lo modificaremos y sobrescribiremos
    # No podemos usar el contexto 'with' de arriba si vamos a escribir sobre el mismo archivo.
    hdul_final = fits.open('modulus_tracker.fits', mode='update')
    
    # Convertir la astropy.table modificada de vuelta a un objeto BinTableHDU
    # y reemplazar el HDU original en la lista
    nuevo_hdu = fits.table_to_hdu(tabla_modificada)
    
    # Asegúrate de mantener los metadatos importantes del header original si es necesario, 
    # como EXTNAME y TELESCOP, que se sobrescriben al usar fits.table_to_hdu()
    nuevo_hdu.header['EXTNAME'] = Galaxy
    nuevo_hdu.header['METHOD'] = Method
    
    # Reemplazar el HDU antiguo por el nuevo en la lista
    hdul_final[hdu_index] = nuevo_hdu
    
    # Escribir los cambios al disco y cerrar el archivo
    hdul_final.flush() # Fuerza la escritura
    hdul_final.close()
    
    print(f"Archivo '{'modulus_tracker.fits'}' actualizado exitosamente con {len(tabla_modificada)} filas.")

    # Verificar leyendo el archivo actualizado
    t_verif = Table.read('modulus_tracker.fits', hdu=hdu_index)
    print("\nVerificación de la tabla en el archivo:")
    print(t_verif)

else:
    print("No se pudo realizar la modificación porque no se encontró el HDU adecuado.")

Buscando HDU con EXTNAME=MRK0116 Y METHOD=CEPHEIDS...
HDU encontrado en el índice 11. Cargando datos para modificar.

Guardando cambios...
Archivo 'modulus_tracker.fits' actualizado exitosamente con 1 filas.

Verificación de la tabla en el archivo:
      RefCode       ...
------------------- ...
2007ApJ...667L.151A ...


In [274]:
S = fits.open('modulus_tracker.fits')

S[11].header

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 1005 / length of dimension 1                          
NAXIS2  =                    0 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                   14 / number of table fields                         
METHOD  = 'CEPHEIDS'           / Cepheids PLR method                            
EXTNAME = 'MRK0116 '           / Name of Host Galaxy                            
TTYPE1  = 'RefCode '           / Article's RefCode                              
TFORM1  = '40A     '                                                            
TTYPE2  = 'Citation'        